# 06 · Binary search

After this notebook you can write binary search without off-by-one bugs using one invariant-based template, use Python's `bisect` module for first/last occurrence and insertion points, and recognise the less obvious uses: rotated arrays, integer square root, peak finding, and **binary search on the answer** (Koko eating bananas, ship capacity).

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · Big-O and timing](01_big_o_and_timing.ipynb) · **Theory:** [DSA basics course](../../dsa_basics/index.html), chapter 10 "Sorting & searching"

## Why this matters in interviews

- Binary search is simple to describe and notoriously easy to get wrong (infinite loops, off-by-one). Interviewers use it to see whether you reason with invariants or by trial and error.
- The big idea generalises far beyond sorted arrays: any **monotone yes/no question** over a range — "can we finish at speed k?" — can be binary-searched in O(log range) checks.
- log₂(10⁹) ≈ 30: a constraint like "answer up to 10⁹" is often a hint that you should binary-search it.

## What interviewers ask

- "Binary search in a sorted array." (LeetCode 704) · "Where would the target be inserted?" (LC 35)
- "First and last position of a value in a sorted array." (LC 34)
- "Search in a rotated sorted array." (LC 33) · "Minimum of a rotated sorted array." (LC 153)
- "Integer square root without `math.sqrt`." (LC 69)
- "Find a peak element in O(log n)." (LC 162)
- "Minimum speed to eat all bananas in h hours." (LC 875) · "Minimum ship capacity to ship everything in d days." (LC 1011)
- Follow-ups: "Why does your loop terminate?" · "What is your invariant?" · "Why is it `lo < hi` and not `lo <= hi`?"

In [ ]:
import bisect
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np

random.seed(0)
rng = random.Random(1)
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=COLORS), "axes.grid": True, "grid.color": "#e1e0d9",
                     "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2})

## The pattern: find the boundary of a monotone predicate

**In plain English:** binary search does not really search for a value — it finds the **boundary** where a yes/no question flips from "no" to "yes". Ask the question about the middle of the range; the answer tells you which half the boundary is in; throw the other half away. Halving each step means about log₂ n questions.

```
 index:    0     1     2     3     4     5     6     7
 pred:     F     F     F     F     T     T     T     T        pred(i) = nums[i] >= target
                             lo=hi=4  <- the first True
 invariant:  everything left of lo is F,  everything from hi onward is T
```

**Recognise it when you see:** a sorted array; "first / last / smallest / largest such that..."; a feasibility check that gets easier as a parameter grows (speed, capacity, time, distance); a huge answer range (10⁹) with a cheap check.

**The one template** — half-open range `[lo, hi)`, returns the first index where `pred` is True (or `hi` if none):

```python
def first_true(lo, hi, pred):
    while lo < hi:                  # the range [lo, hi) still has candidates
        mid = (lo + hi) // 2        # lo <= mid < hi, so both updates shrink the range
        if pred(mid):
            hi = mid                # mid might be the answer: keep it
        else:
            lo = mid + 1            # mid is not: discard it
    return lo                       # lo == hi: the boundary
```

Why it cannot loop forever: `mid < hi` always, so `hi = mid` shrinks the range, and `lo = mid + 1` shrinks it too. The classic bug is `lo = mid`: when `hi = lo + 1`, `mid == lo` and nothing changes.

In [ ]:
def first_true(lo, hi, pred, trace=None):
    """Smallest i in [lo, hi) with pred(i) True, or hi if there is none. pred must be F...F T...T."""
    while lo < hi:
        mid = (lo + hi) // 2
        if trace is not None:
            trace.append((lo, hi, mid))
        if pred(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


def buggy_first_true(lo, hi, pred, max_steps=50):
    """The classic bug: lo = mid instead of lo = mid + 1."""
    for _ in range(max_steps):
        if lo >= hi:
            return lo
        mid = (lo + hi) // 2
        if pred(mid):
            hi = mid
        else:
            lo = mid                    # BUG: when hi == lo + 1, mid == lo and nothing moves
    return "stuck"


assert first_true(0, 8, lambda i: i >= 4) == 4
assert first_true(0, 8, lambda i: False) == 8 and first_true(0, 8, lambda i: True) == 0 and first_true(0, 0, lambda i: True) == 0
print("buggy version on an all-False range:", buggy_first_true(0, 8, lambda i: False))
assert buggy_first_true(0, 8, lambda i: False) == "stuck"
for _ in range(300):                                # every boundary position, every range size
    n = rng.randint(0, 40)
    boundary = rng.randint(0, n)
    assert first_true(0, n, lambda i: i >= boundary) == boundary
print("first_true finds every boundary in ranges of size 0-40; the lo = mid version gets stuck")

A run of the template on a sorted array, looking for the first element ≥ 23. Each row is one question; the grey cells are already ruled out:

In [ ]:
nums = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91, 95, 99]
target = 23
steps = []
found = first_true(0, len(nums), lambda i: nums[i] >= target, steps)
steps.append((found, found, None))

fig, ax = plt.subplots(figsize=(10, 0.6 * len(steps) + 0.9))
for row, (lo, hi, mid) in enumerate(steps):
    y = len(steps) - 1 - row
    for i, v in enumerate(nums):
        if i == mid:
            face, ink = COLORS[1], "white"
        elif lo <= i < hi:
            face, ink = "#b7d3f6", "#0b0b0b"
        elif mid is None and i == found:
            face, ink = COLORS[2], "white"
        else:
            face, ink = "#f0efec", "#898781"
        ax.add_patch(plt.Rectangle((i, y), 0.92, 0.8, facecolor=face, edgecolor="none"))
        ax.text(i + 0.46, y + 0.4, str(v), ha="center", va="center", color=ink, fontsize=10)
    note = (f"lo={lo} hi={hi} mid={mid}: {nums[mid]} >= {target}? {'yes -> hi = mid' if nums[mid] >= target else 'no -> lo = mid + 1'}"
            if mid is not None else f"lo == hi == {found}: first element >= {target}")
    ax.text(len(nums) + 0.3, y + 0.4, note, va="center", fontsize=9)
ax.set_xlim(-0.2, len(nums) + 9)
ax.set_ylim(-0.2, len(steps))
ax.axis("off")
ax.set_title(f"first_true on 12 sorted numbers: {len(steps) - 1} questions (log₂ 12 ≈ 3.6), mid in orange")
plt.show()
assert nums[found] == target

## Problem 1 · Binary search and search insert position (LC 704, LC 35)

**Statement:** in a sorted array of distinct integers, return the index of `target` or −1 (LC 704) — or the index where it *would be inserted* to keep the order (LC 35). `[-1,0,3,5,9,12]`, 9 → 4; `[1,3,5,6]`, 2 → insert at 1.

**Brute force:** a linear scan, O(n). **Optimal:** both are the same boundary — the first index with `nums[i] >= target`. You will also meet the closed-interval `[lo, hi]` style with `while lo <= hi`; it is correct too, just keep its updates (`mid ± 1`) consistent.

In [ ]:
def search(nums, target):
    i = first_true(0, len(nums), lambda i: nums[i] >= target)
    return i if i < len(nums) and nums[i] == target else -1


def search_insert(nums, target):
    return first_true(0, len(nums), lambda i: nums[i] >= target)


def search_closed(nums, target):
    """The other common style: closed range [lo, hi], loop while lo <= hi."""
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1


assert search([-1, 0, 3, 5, 9, 12], 9) == 4 and search([-1, 0, 3, 5, 9, 12], 2) == -1
assert [search_insert([1, 3, 5, 6], t) for t in (5, 2, 7, 0)] == [2, 1, 4, 0]
assert search([], 3) == -1 and search_insert([], 3) == 0
for _ in range(500):
    nums = sorted(rng.sample(range(-50, 50), rng.randint(0, 20)))
    t = rng.randint(-55, 55)
    expected = nums.index(t) if t in nums else -1
    assert search(nums, t) == search_closed(nums, t) == expected
    assert search_insert(nums, t) == sum(x < t for x in nums)
print("search, search_closed and search_insert agree with linear scans on 500 random arrays")

## The `bisect` module: binary search you don't have to write

| Call | Returns | Use it for |
|---|---|---|
| `bisect_left(a, x)` | first index `i` with `a[i] >= x` | lower bound, first occurrence, insertion point before equals |
| `bisect_right(a, x)` (= `bisect`) | first index `i` with `a[i] > x` | upper bound, one past the last occurrence |
| `bisect_right(a, x) - bisect_left(a, x)` | number of copies of `x` | counting in O(log n) |
| `bisect_left(a, x, lo, hi)` | the same, inside `a[lo:hi]` | searching part of a list without slicing |
| `bisect_left(a, x, key=f)` (Python 3.10+) | compare `f(a[i])` with `x` | lists of records sorted by a field |
| `insort(a, x)` | inserts in order | **O(n)**: the search is O(log n), the insert shifts elements |

In [ ]:
a = [1, 2, 4, 4, 4, 7, 9]
print("a =", a)
print("bisect_left(a, 4) =", bisect.bisect_left(a, 4), "| bisect_right(a, 4) =", bisect.bisect_right(a, 4),
      "| copies of 4 =", bisect.bisect_right(a, 4) - bisect.bisect_left(a, 4))
print("insertion points for 0, 5, 10:", [bisect.bisect_left(a, x) for x in (0, 5, 10)])


def search_range(nums, target):
    """LC 34: first and last index of target, or [-1, -1]. Two binary searches: O(log n)."""
    first = bisect.bisect_left(nums, target)
    if first == len(nums) or nums[first] != target:
        return [-1, -1]
    return [first, bisect.bisect_right(nums, target) - 1]


def grade(score, breakpoints=(60, 70, 80, 90), grades="FDCBA"):
    return grades[bisect.bisect_right(breakpoints, score)]     # the example from the bisect docs


records = [("ana", 55), ("bo", 70), ("cy", 70), ("di", 88), ("ed", 93)]      # sorted by score
first_70 = bisect.bisect_left(records, 70, key=lambda r: r[1])
print("first record with score >= 70:", records[first_70], "| grades for 59, 60, 85, 90:", [grade(s) for s in (59, 60, 85, 90)])

assert search_range([5, 7, 7, 8, 8, 10], 8) == [3, 4] and search_range([5, 7, 7, 8, 8, 10], 6) == [-1, -1]
assert search_range([], 0) == [-1, -1] and search_range([1], 1) == [0, 0]
assert [grade(s) for s in (59, 60, 85, 90)] == ["F", "D", "B", "A"] and records[first_70] == ("bo", 70)
for _ in range(300):
    nums = sorted(rng.randint(0, 8) for _ in range(rng.randint(0, 15)))
    t = rng.randint(0, 9)
    expected = [nums.index(t), len(nums) - 1 - nums[::-1].index(t)] if t in nums else [-1, -1]
    assert search_range(nums, t) == expected
print("search_range (LC 34) agrees with a linear scan on 300 random arrays with duplicates")

## Problem 2 · Search in a rotated sorted array (LC 33)

**Statement:** a sorted array of distinct values was rotated at an unknown pivot, e.g. `[4,5,6,7,0,1,2]`. Find `target`'s index in O(log n), or −1.

**Brute force:** `nums.index(target)` — O(n). **Optimal (one pass):** at every step *one half is sorted* — compare `nums[lo]` with `nums[mid]` to see which. If the target lies inside the sorted half's range, search there; otherwise search the other half.

**Optimal (two steps, easier to get right):** the minimum is the first index where `nums[i] <= nums[-1]` — a monotone predicate — so `first_true` finds the pivot; then run an ordinary `bisect` on the correct sorted part.

In [ ]:
def search_rotated(nums, target):
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[lo] <= nums[mid]:                       # the left half nums[lo..mid] is sorted
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:                                           # the right half nums[mid..hi] is sorted
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1


def search_rotated_two_step(nums, target):
    if not nums:
        return -1
    pivot = first_true(0, len(nums), lambda i: nums[i] <= nums[-1])    # index of the minimum
    lo, hi = (pivot, len(nums)) if target <= nums[-1] else (0, pivot)
    i = bisect.bisect_left(nums, target, lo, hi)
    return i if i < hi and nums[i] == target else -1


for nums, t, expected in [([4, 5, 6, 7, 0, 1, 2], 0, 4), ([4, 5, 6, 7, 0, 1, 2], 3, -1), ([1], 0, -1), ([1], 1, 0),
                          ([3, 1], 1, 1), ([1, 3], 3, 1), ([5, 1, 3], 5, 0), ([], 5, -1)]:
    assert search_rotated(nums, t) == search_rotated_two_step(nums, t) == expected, (nums, t)
for _ in range(500):
    base = sorted(rng.sample(range(100), rng.randint(1, 15)))
    k = rng.randrange(len(base))
    nums = base[k:] + base[:k]                          # rotate
    t = rng.randint(0, 100)
    expected = nums.index(t) if t in nums else -1
    assert search_rotated(nums, t) == search_rotated_two_step(nums, t) == expected
print("both rotated-array searches agree with nums.index on 500 random rotations")

**Complexity:** O(log n) time, O(1) space. **Trap:** `nums[lo] <= nums[mid]` needs the `=` — with two elements `lo == mid`. With duplicates allowed (LC 81), `nums[lo] == nums[mid]` tells you nothing and the worst case degrades to O(n).

## Problem 3 · Integer square root (LC 69)

**Statement:** return ⌊√n⌋ without `math.sqrt` or `**0.5`. 8 → 2, 16 → 4.

**Brute force:** try x = 0, 1, 2, … while x² ≤ n — O(√n), about 3·10⁴ steps for n = 10⁹. **Optimal:** `x * x > n` is monotone in x, so the answer is `first_true(x * x > n) − 1` — about log₂ n ≈ 30 steps. Integer arithmetic also avoids float rounding for huge n.

In [ ]:
def isqrt(n):
    return first_true(0, n + 1, lambda x: x * x > n) - 1


for n, expected in [(0, 0), (1, 1), (4, 2), (8, 2), (15, 3), (16, 4), (2147395599, 46339)]:
    assert isqrt(n) == expected == math.isqrt(n), n
for _ in range(1_000):
    n = rng.randint(0, 10 ** 12)
    assert isqrt(n) == math.isqrt(n)
steps = []
first_true(0, 10 ** 12 + 1, lambda x: x * x > 10 ** 12, steps)
print(f"isqrt matches math.isqrt on 1,000 random n up to 10^12; n = 10^12 took {len(steps)} steps (log2 = {math.log2(10 ** 12):.1f})")
assert len(steps) <= 41

## Problem 4 · Find a peak element (LC 162)

**Statement:** return the index of any element strictly greater than its neighbours (treat `nums[-1]` and `nums[n]` as −∞; no two neighbours are equal). `[1,2,1,3,5,6,4]` → 1 or 5.

**Brute force:** scan for it — O(n). **Optimal:** the array is not sorted, but compare `nums[mid]` with `nums[mid + 1]`: if the slope goes *down*, a peak exists at `mid` or to its left; if it goes *up*, a peak exists strictly to the right (keep climbing — you either find a drop or hit the −∞ edge). The invariant is "a peak lies in `[lo, hi]`", which is enough even though the predicate is not globally monotone.

In [ ]:
def find_peak(nums):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[mid + 1]:
            hi = mid                 # downhill: a peak at mid or to the left
        else:
            lo = mid + 1             # uphill: a peak to the right
    return lo


def is_peak(nums, i):
    left = nums[i - 1] if i > 0 else -math.inf
    right = nums[i + 1] if i + 1 < len(nums) else -math.inf
    return left < nums[i] > right


assert find_peak([1, 2, 3, 1]) == 2 and find_peak([1, 2, 1, 3, 5, 6, 4]) in (1, 5)
assert find_peak([1]) == 0 and find_peak([1, 2]) == 1 and find_peak([2, 1]) == 0 and find_peak(list(range(10))) == 9
for _ in range(500):
    nums = rng.sample(range(100), rng.randint(1, 20))           # distinct values: no equal neighbours
    assert is_peak(nums, find_peak(nums))
print("find_peak returns a genuine peak on 500 random arrays in O(log n)")

## Problem 5 · Binary search on the answer: Koko eating bananas (LC 875)

**Statement:** Koko eats from one pile per hour, at most `k` bananas from it; a pile of size `p` takes ⌈p / k⌉ hours. What is the minimum `k` that finishes all piles within `h` hours? `piles = [3,6,7,11]`, h = 8 → 4.

**The shift in thinking:** don't search the array — search the **answer space**. "Can she finish at speed k?" is monotone: if speed k works, every faster speed works too. So the answer is `first_true` over k in `[1, max(piles)]`, with an O(n) check per question: O(n log max) total. **Brute force:** try k = 1, 2, 3, … — O(n · max).

In [ ]:
def hours_needed(piles, speed):
    return sum((p + speed - 1) // speed for p in piles)          # ceil(p / speed) without floats


def min_eating_speed(piles, h):
    return first_true(1, max(piles) + 1, lambda k: hours_needed(piles, k) <= h)


def min_eating_speed_brute(piles, h):
    return next(k for k in range(1, max(piles) + 1) if hours_needed(piles, k) <= h)


for (piles, h), expected in [(([3, 6, 7, 11], 8), 4), (([30, 11, 23, 4, 20], 5), 30), (([30, 11, 23, 4, 20], 6), 23),
                             (([1], 1), 1), (([1_000_000_000], 2), 500_000_000)]:
    assert min_eating_speed(piles, h) == expected, (piles, h)
for _ in range(300):
    piles = [rng.randint(1, 40) for _ in range(rng.randint(1, 6))]
    h = rng.randint(len(piles), 60)                               # h >= number of piles, so an answer exists
    assert min_eating_speed(piles, h) == min_eating_speed_brute(piles, h)
print("min_eating_speed agrees with trying every speed; the 10^9 pile needs only ~30 checks")

In [ ]:
piles, h = [30, 11, 23, 4, 20], 6
speeds = np.arange(1, max(piles) + 1)
hours = [hours_needed(piles, k) for k in speeds]
answer = min_eating_speed(piles, h)
fig, ax = plt.subplots(figsize=(9, 4))
ax.step(speeds, hours, where="mid", color=COLORS[0], label="hours needed at speed k")
ax.axhline(h, color=COLORS[1], lw=1.5, label=f"limit h = {h}")
ax.axvspan(answer - 0.5, speeds[-1] + 0.5, color=COLORS[2], alpha=0.12, label="feasible speeds (monotone!)")
ax.plot([answer], [hours_needed(piles, answer)], "o", ms=9, color=COLORS[2])
ax.annotate(f"answer: k = {answer}", xy=(answer, hours_needed(piles, answer)), xytext=(answer - 12, h + 12),
            arrowprops=dict(arrowstyle="->", lw=1.2), fontsize=10)
ax.set(title=f"Koko, piles {piles}: hours never increase as speed grows", xlabel="speed k (bananas per hour)",
       ylabel="hours", yscale="log")
ax.legend()
plt.show()

## Problem 6 · Capacity to ship packages within D days (LC 1011)

**Statement:** packages must ship **in order**; each day the ship carries a contiguous run whose total weight is ≤ its capacity. Find the minimum capacity that ships everything within `days`. `[1..10]`, days = 5 → 15.

**Same pattern:** the search range is `[max(weights), sum(weights)]` (it must carry the heaviest package; carrying everything in one day always works). The check is a greedy pass: fill each day until the next package would not fit. More capacity never needs more days, so the check is monotone.

In [ ]:
def days_needed(weights, capacity):
    days, load = 1, 0
    for w in weights:
        if load + w > capacity:               # start a new day
            days, load = days + 1, 0
        load += w
    return days


def ship_within_days(weights, days):
    return first_true(max(weights), sum(weights) + 1, lambda cap: days_needed(weights, cap) <= days)


def ship_within_days_brute(weights, days):
    return next(c for c in range(max(weights), sum(weights) + 1) if days_needed(weights, c) <= days)


for (weights, d), expected in [((list(range(1, 11)), 5), 15), (([3, 2, 2, 4, 1, 4], 3), 6), (([1, 2, 3, 1, 1], 4), 3), (([7], 1), 7)]:
    assert ship_within_days(weights, d) == expected, (weights, d)
for _ in range(300):
    weights = [rng.randint(1, 15) for _ in range(rng.randint(1, 10))]
    d = rng.randint(1, len(weights))
    assert ship_within_days(weights, d) == ship_within_days_brute(weights, d)
print("ship_within_days agrees with a linear scan over capacities on 300 random inputs")

The same shape answers "split an array into k parts minimising the largest sum" (LC 410), "minimum days to make m bouquets" (LC 1482) and "place k routers as far apart as possible" (maximise the minimum distance: search for the *last* feasible value). Recipe: **(1)** find the answer range, **(2)** write `feasible(x)`, **(3)** check it is monotone, **(4)** `first_true`.

## How much faster is O(log n)?

Timing `list.index` (a C-level linear scan) against `bisect_left` (a C-level binary search) for the last element of sorted lists of growing size:

In [ ]:
def interleaved_times(calls, number=1, rounds=3):
    best = [float("inf")] * len(calls)
    for _ in range(rounds):
        for i, call in enumerate(calls):
            t0 = time.perf_counter()
            for _ in range(number):
                call()
            best[i] = min(best[i], (time.perf_counter() - t0) / number)
    return best


sizes = [2 ** k for k in range(10, 20)]                 # 1,024 ... 524,288
data = list(range(sizes[-1]))
t_linear = interleaved_times([lambda n=n: data.index(n - 1, 0, n) for n in sizes], number=3)     # worst case: last element
t_binary = interleaved_times([lambda n=n: bisect.bisect_left(data, n - 1, 0, n) for n in sizes], number=2_000)
slope_linear = np.polyfit(np.log2(sizes), np.log2(t_linear), 1)[0]
slope_binary = np.polyfit(np.log2(sizes), np.log2(t_binary), 1)[0]

fig, ax = plt.subplots(figsize=(9, 4))
ax.loglog(sizes, np.array(t_linear) * 1e6, marker="o", label=f"list.index — linear search (slope {slope_linear:.2f})")
ax.loglog(sizes, np.array(t_binary) * 1e6, marker="o", label=f"bisect_left — binary search (slope {slope_binary:.2f})")
ax.set(title="Searching a sorted list: O(n) vs O(log n)", xlabel="n (log scale)", ylabel="time per search, µs (log scale)")
ax.legend()
plt.show()
print(f"at n = {sizes[-1]:,}: linear {t_linear[-1] * 1e6:,.0f} µs vs binary {t_binary[-1] * 1e6:.2f} µs "
      f"({t_linear[-1] / t_binary[-1]:,.0f}x); comparisons: {sizes[-1]:,} vs {math.ceil(math.log2(sizes[-1]))}")
assert slope_linear > 0.7 and slope_binary < 0.35 and t_binary[-1] * 100 < t_linear[-1]

The catch: binary search needs sorted data, and sorting costs O(n log n). It pays off when you search the same data many times (k searches: O(n log n + k log n) instead of O(k·n)), or when the "array" is implicit — a range of answers that never has to be built at all.

## Pattern cheat sheet — if you see X, think Y

| If you see... | Think... |
|---|---|
| sorted array: find / insert / count | `bisect_left`, `bisect_right` |
| "first position", lower bound | `bisect_left` or `first_true(nums[i] >= x)` |
| "last position", upper bound | `bisect_right(...) - 1` |
| "minimum X such that feasible(X)", feasibility monotone in X | binary search on the answer: `first_true(lo, hi, feasible)` |
| "maximum X such that ..." | the last True = `first_true(not feasible) - 1` |
| rotated sorted array | one half is always sorted; or find the pivot with `nums[i] <= nums[-1]` |
| peak / local maximum in O(log n) | compare `mid` with `mid + 1`, climb uphill |
| real-valued answer to a precision | loop a fixed number of times (e.g. 100) on floats |
| 2-D matrix sorted row after row | treat it as one array: `row, col = divmod(i, cols)` |
| answer or values up to 10⁹, cheap O(n) check | O(n log 10⁹) ≈ 30 checks |

## Try it yourself

**1. Minimum in a rotated sorted array (LC 153).** `[3,4,5,1,2]` → 1, `[11,13,15,17]` → 11. One `first_true` call.

In [ ]:
# Solution — try it yourself first, then run this
def find_min_rotated(nums):
    return nums[first_true(0, len(nums), lambda i: nums[i] <= nums[-1])]


for nums in [[3, 4, 5, 1, 2], [4, 5, 6, 7, 0, 1, 2], [11, 13, 15, 17], [2, 1], [1]]:
    assert find_min_rotated(nums) == min(nums), nums
for _ in range(300):
    base = sorted(rng.sample(range(100), rng.randint(1, 12)))
    k = rng.randrange(len(base))
    assert find_min_rotated(base[k:] + base[:k]) == base[0]
print("find_min_rotated: all rotations pass")

**2. First bad version (LC 278).** Versions `1..n`; once a version is bad, all later ones are bad. Find the first bad one using as few calls to `is_bad(v)` as possible.

In [ ]:
# Solution — try it yourself first, then run this
def first_bad_version(n, is_bad):
    return first_true(1, n + 1, is_bad)


calls = []
first_bad = 1_234_567


def is_bad(v):
    calls.append(v)
    return v >= first_bad


assert first_bad_version(2_000_000_000, is_bad) == first_bad
print(f"found version {first_bad:,} among 2 billion with {len(calls)} calls to is_bad")
assert len(calls) <= 31

**3. Search a 2-D matrix (LC 74).** Each row is sorted and each row starts after the previous row ends. Return whether `target` is present, in O(log(rows × cols)).

In [ ]:
# Solution — try it yourself first, then run this
def search_matrix(matrix, target):
    if not matrix or not matrix[0]:
        return False
    rows, cols = len(matrix), len(matrix[0])
    i = first_true(0, rows * cols, lambda k: matrix[k // cols][k % cols] >= target)   # a virtual 1-D array
    return i < rows * cols and matrix[i // cols][i % cols] == target


m = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]]
assert search_matrix(m, 3) and not search_matrix(m, 13) and search_matrix(m, 60) and not search_matrix(m, 0)
assert not search_matrix([], 1) and not search_matrix([[]], 1)
assert all(search_matrix(m, v) for row in m for v in row)
print("search_matrix: every stored value is found, absent values are not")

**4. Real-valued cube root.** Compute ∛x for x ≥ 0 to within 1e-9 by bisection on floats. With floats, don't loop on `lo < hi` — loop a fixed number of times; each iteration halves the interval.

In [ ]:
# Solution — try it yourself first, then run this
def cube_root(x, iterations=100):
    lo, hi = 0.0, max(1.0, x)                 # the root lies in [0, max(1, x)]
    for _ in range(iterations):               # 100 halvings: far below float precision
        mid = (lo + hi) / 2
        if mid ** 3 < x:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2


for x in [0.0, 0.001, 1.0, 2.0, 27.0, 1e6, 12345.678]:
    assert abs(cube_root(x) - x ** (1 / 3)) < 1e-9, x
print("cube_root(2) =", f"{cube_root(2.0):.12f}", "| cube_root(0.001) =", f"{cube_root(0.001):.12f}")

## Say it in an interview

- **30-second answer:** "Binary search finds the boundary of a monotone yes/no predicate by halving the candidate range, so it needs about log₂ n questions. I keep an invariant — everything left of `lo` is False, everything from `hi` on is True — use a half-open range with `lo = mid + 1` and `hi = mid`, and stop when `lo == hi`."
- **Binary search on the answer:** "If a feasibility check is monotone in a parameter — faster speed, bigger capacity — I search the parameter's range instead of an array: O(check × log range)."
- **Key numbers:** log₂(10⁶) ≈ 20, log₂(10⁹) ≈ 30, log₂(10¹⁸) ≈ 60. `bisect_left`/`bisect_right` are O(log n); `insort` is O(n).
- **Follow-ups:** "Duplicates?" — `bisect_left` vs `bisect_right` decide first vs last. "Rotated with duplicates?" — worst case O(n). "Why not `(lo + hi) // 2` in Java?" — overflow; use `lo + (hi - lo) // 2`. Python ints don't overflow.
- **Traps:** `lo = mid` (infinite loop); mixing the closed `[lo, hi]` and half-open `[lo, hi)` styles; forgetting the "not found" check after `bisect_left` (`i < len(a) and a[i] == x`); a non-monotone feasibility check (then binary search is simply wrong).

## Next

- [07 · Sorting algorithms](07_sorting_algorithms.ipynb) — the O(n log n) that makes binary search possible
- [09 · Trees and BST](09_trees_and_bst.ipynb) — binary search as a data structure
- [02 · Arrays and two pointers](02_arrays_and_two_pointers.ipynb) — the other way to exploit sorted data
- GenAI link: [Vector search with FAISS and HNSW](../09_rag/03_vector_search_faiss_hnsw.ipynb) — "don't look at everything", in high dimensions
- Theory: [DSA basics course](../../dsa_basics/index.html) (chapter 10, "Sorting & searching") · [interview bank](../../ai_interview_prep/index.html)